# Clasificador de Motivo de Reclamación

Pipeline completo de Machine Learning para clasificar el motivo de reclamación usando XGBoost.

**Pasos:**
1. Configuración y carga de datos
2. Análisis Exploratorio (EDA)
3. Feature Engineering
4. Entrenamiento del modelo base
5. Optimización de hiperparámetros
6. Evaluación del modelo
7. Guardado del modelo

## 1. Configuración e Instalación de Dependencias

In [ ]:
# Instalar dependencias (descomentar si es necesario)
 ! pip install pandas numpy scikit-learn xgboost joblib matplotlib seaborn

: 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import joblib
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

print("Dependencias cargadas correctamente.")

In [ ]:
# === CONFIGURACIÓN ===

DATA_PATH = "quejas_limpias.csv"
MODEL_PATH = "clasificador-reclamos-backend/models/model.pkl"
METRICS_PATH = "clasificador-reclamos-backend/models/metrics.json"

CATEGORICAL_FEATURES = [
    "SECTOR",
    "TIPO_RECLAMACION",
    "PROCEDIMIENTO",
    "BIEN O SERV",
    "MEDIO INGRESO",
    "TIPO PROD",
    "MODALIDAD COMPRA",
    "MODALIDAD PAGO",
    "PROB ESPECIAL",
]

NUMERICAL_FEATURES = [
    "COSTO BIEN SERVICIO",
    "MONTO RECLAMADO",
    "MONTO RECUPERADO",
    "DIAS_RESOLUCION",
]

TARGET = "MOTIVO_RECLAMACION"

EXCLUDE_COLUMNS = ["ID_EXP", "PROVEEDOR", "FECHA_INGRESO", "FECHA_FIN", "FECHA DE CIERRE"]

MIN_CATEGORY_COUNT = 30
TEST_SIZE = 0.2
RANDOM_STATE = 42

EXPECTED_COLUMNS = [
    "ID_EXP", "FECHA_INGRESO", "FECHA_FIN", "FECHA DE CIERRE", "PROVEEDOR",
    "SECTOR", "TIPO_RECLAMACION", "MOTIVO_RECLAMACION", "COSTO BIEN SERVICIO",
    "MONTO RECLAMADO", "MONTO RECUPERADO", "PROCEDIMIENTO", "BIEN O SERV",
    "MEDIO INGRESO", "TIPO PROD", "MODALIDAD COMPRA", "MODALIDAD PAGO", "PROB ESPECIAL",
]

print("Configuración lista.")

## 2. Carga de Datos

In [ ]:
# Cargar dataset
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# Validar columnas esperadas
missing = set(EXPECTED_COLUMNS) - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas en el dataset: {missing}")

print(f"Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Distribución del target
print(f"Categorías únicas del target: {df[TARGET].nunique()}")
print(f"\nDistribución de '{TARGET}':")

counts = df[TARGET].value_counts()
for cat, count in counts.items():
    pct = count / len(df) * 100
    print(f"  {cat}: {count} ({pct:.1f}%)")

In [ ]:
# Gráfico de distribución del target
fig, ax = plt.subplots(figsize=(12, 6))
counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Distribución de {TARGET}')
ax.set_xlabel('Cantidad de registros')
plt.tight_layout()
plt.show()

In [ ]:
# Valores nulos
print("Porcentaje de valores nulos por columna:")
null_pct = df.isnull().mean() * 100
for col, pct in null_pct.items():
    if pct > 0:
        print(f"  {col}: {pct:.2f}%")

if null_pct.sum() == 0:
    print("  (sin valores nulos)")

In [ ]:
# Categorías con pocos registros
low_count = counts[counts < MIN_CATEGORY_COUNT]
if len(low_count) > 0:
    print(f"Categorías con menos de {MIN_CATEGORY_COUNT} registros:")
    for cat, count in low_count.items():
        print(f"  {cat}: {count}")
else:
    print(f"Todas las categorías tienen >= {MIN_CATEGORY_COUNT} registros.")

In [ ]:
# Estadísticas descriptivas de features numéricas
numerical_cols = ["COSTO BIEN SERVICIO", "MONTO RECLAMADO", "MONTO RECUPERADO"]
df[numerical_cols].describe()

## 4. Limpieza de Datos

In [ ]:
# Eliminar duplicados por ID_EXP
dupes = df.duplicated(subset=["ID_EXP"], keep="first").sum()
if dupes > 0:
    print(f"Eliminando {dupes} registros duplicados por ID_EXP.")
    df = df.drop_duplicates(subset=["ID_EXP"], keep="first").reset_index(drop=True)
else:
    print("Sin registros duplicados por ID_EXP.")

print(f"Registros tras eliminar duplicados: {len(df)}")

In [ ]:
# Agrupar categorías con pocos registros bajo "Otros"
counts = df[TARGET].value_counts()
low_cats = counts[counts < MIN_CATEGORY_COUNT].index.tolist()

if low_cats:
    print(f"Agrupando {len(low_cats)} categorías con < {MIN_CATEGORY_COUNT} registros bajo 'Otros':")
    for cat in low_cats:
        print(f"  - {cat} ({counts[cat]} registros)")
    df[TARGET] = df[TARGET].apply(lambda x: "Otros" if x in low_cats else x)
else:
    print("No hay categorías para agrupar.")

print(f"\nDataset final: {len(df)} registros, {df[TARGET].nunique()} categorías.")

## 5. Feature Engineering

In [ ]:
# Crear feature temporal: DIAS_RESOLUCION
df["DIAS_RESOLUCION"] = (
    pd.to_datetime(df["FECHA_FIN"], errors="coerce")
    - pd.to_datetime(df["FECHA_INGRESO"], errors="coerce")
).dt.days

print("Feature DIAS_RESOLUCION creada.")
print(f"  Min: {df['DIAS_RESOLUCION'].min()}, Max: {df['DIAS_RESOLUCION'].max()}, Media: {df['DIAS_RESOLUCION'].mean():.1f}")

In [ ]:
# Tratar valores nulos

# Categóricas: nulo -> "Desconocido"
for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna("Desconocido")

# Numéricas: nulo -> mediana
for col in NUMERICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

print("Valores nulos tratados.")
print(f"Nulos restantes: {df[CATEGORICAL_FEATURES + NUMERICAL_FEATURES].isnull().sum().sum()}")

In [ ]:
# Codificar variables categóricas con LabelEncoder
encoders = {}

for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"  {col}: {len(le.classes_)} clases")

# Codificar target
target_encoder = LabelEncoder()
df[TARGET] = target_encoder.fit_transform(df[TARGET])

print(f"\nVariables categóricas codificadas: {len(encoders)}")
print(f"Categorías del target: {target_encoder.classes_.tolist()}")

In [ ]:
# Escalar features numéricas
scaler = StandardScaler()
df[NUMERICAL_FEATURES] = scaler.fit_transform(df[NUMERICAL_FEATURES])

print("Features numéricas escaladas.")

## 6. Preparar Conjuntos de Entrenamiento y Test

In [ ]:
# Construir X e y
feature_cols = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
X = df[feature_cols]
y = df[TARGET]

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Split: {len(X_train)} train / {len(X_test)} test")
print(f"Features: {len(feature_cols)} ({len(CATEGORICAL_FEATURES)} categóricas + {len(NUMERICAL_FEATURES)} numéricas)")

In [ ]:
# Calcular pesos de muestra para balanceo de clases
sample_weights = compute_sample_weight("balanced", y_train)

print("Sample weights calculados para balanceo de clases.")

In [ ]:
# Guardar artefactos para uso posterior
artifacts = {
    "encoders": encoders,
    "target_encoder": target_encoder,
    "scaler": scaler,
    "feature_names": feature_cols,
    "categorical_features": CATEGORICAL_FEATURES,
    "numerical_features": NUMERICAL_FEATURES,
    "categories": target_encoder.classes_.tolist(),
}

print("Artefactos almacenados.")

## 7. Entrenamiento del Modelo Base

In [ ]:
base_model = xgb.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

base_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

# Evaluación rápida del modelo base
base_score = base_model.score(X_test, y_test)
print(f"Accuracy del modelo base: {base_score:.4f}")

## 8. Optimización de Hiperparámetros

In [ ]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.2, 0.3],
    "reg_alpha": [0, 0.1, 0.5, 1.0],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0],
}

search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_weighted",
    cv=5,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1,
)

search.fit(X_train, y_train, sample_weight=sample_weights)

best_model = search.best_estimator_
best_params = search.best_params_
best_score = search.best_score_

print(f"\nMejor F1 (CV): {best_score:.4f}")
print(f"Mejores parámetros: {best_params}")

## 9. Validación Cruzada del Mejor Modelo

In [ ]:
cv_scores = cross_val_score(
    best_model, X_train, y_train,
    cv=5,
    scoring="f1_weighted",
)

print(f"F1 promedio (5-fold CV): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Scores individuales: {[f'{s:.4f}' for s in cv_scores]}")

## 10. Evaluación del Modelo

In [ ]:
# Predicciones
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=90, values_format='d')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importance = best_model.feature_importances_
feature_names = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
feature_importance = dict(zip(feature_names, importance.tolist()))
feature_importance = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))

print("Feature Importance:")
for feat, imp in feature_importance.items():
    bar = "#" * int(imp * 100)
    print(f"  {feat:25s} {imp:.4f} {bar}")

# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(feature_importance.keys())[::-1], list(feature_importance.values())[::-1], color='steelblue')
ax.set_title('Importancia de Features')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

## 11. Guardar Métricas

In [ ]:
# Generar reporte de clasificación como dict
report = classification_report(
    y_test, y_pred,
    target_names=target_encoder.classes_,
    output_dict=True,
)

metrics = {
    "classification_report": report,
    "confusion_matrix": cm.tolist(),
    "feature_importance": feature_importance,
    "best_params": best_params,
    "cv_f1_mean": float(cv_scores.mean()),
    "cv_f1_std": float(cv_scores.std()),
    "categories": target_encoder.classes_.tolist(),
}

os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(f"Métricas guardadas en: {METRICS_PATH}")
print(f"F1-Score weighted: {report['weighted avg']['f1-score']:.4f}")
print(f"Accuracy: {report['accuracy']:.4f}")

## 12. Guardar Modelo

In [ ]:
# Empaquetar modelo + artefactos
artifacts["model"] = best_model

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(artifacts, MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")

# Verificar serialización
loaded = joblib.load(MODEL_PATH)
assert loaded["model"] is not None
assert len(loaded["encoders"]) == len(artifacts["encoders"])
print("Modelo serializado y verificado correctamente.")

## 13. Prueba de Predicción

In [ ]:
def predict_claim(input_data: dict, artifacts: dict) -> dict:
    """Función de predicción individual (misma que usa el backend API)."""
    model = artifacts["model"]
    encoders = artifacts["encoders"]
    target_encoder = artifacts["target_encoder"]
    scaler = artifacts["scaler"]
    categorical_features = artifacts["categorical_features"]
    numerical_features = artifacts["numerical_features"]

    key_to_col = {
        "sector": "SECTOR",
        "tipo_reclamacion": "TIPO_RECLAMACION",
        "procedimiento": "PROCEDIMIENTO",
        "bien_o_serv": "BIEN O SERV",
        "medio_ingreso": "MEDIO INGRESO",
        "tipo_prod": "TIPO PROD",
        "modalidad_compra": "MODALIDAD COMPRA",
        "modalidad_pago": "MODALIDAD PAGO",
        "prob_especial": "PROB ESPECIAL",
        "costo_bien_servicio": "COSTO BIEN SERVICIO",
        "monto_reclamado": "MONTO RECLAMADO",
        "monto_recuperado": "MONTO RECUPERADO",
        "dias_resolucion": "DIAS_RESOLUCION",
    }

    row = {}
    for key, col in key_to_col.items():
        if key not in input_data:
            raise ValueError(f"Campo requerido faltante: {key}")
        row[col] = input_data[key]

    for col in categorical_features:
        val = str(row[col])
        le = encoders[col]
        if val in le.classes_:
            row[col] = le.transform([val])[0]
        else:
            row[col] = 0

    num_values = np.array([[row[col] for col in numerical_features]])
    num_scaled = scaler.transform(num_values)[0]
    for i, col in enumerate(numerical_features):
        row[col] = num_scaled[i]

    feature_order = categorical_features + numerical_features
    X = np.array([[row[col] for col in feature_order]])

    proba = model.predict_proba(X)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = target_encoder.inverse_transform([pred_idx])[0]

    probabilidades = {}
    for idx, class_name in enumerate(target_encoder.classes_):
        probabilidades[class_name] = round(float(proba[idx]), 4)

    probabilidades = dict(sorted(probabilidades.items(), key=lambda x: x[1], reverse=True))

    return {
        "prediccion": pred_label,
        "confianza": round(float(proba[pred_idx]), 4),
        "probabilidades": probabilidades,
    }


# Ejemplo de predicción
ejemplo = {
    "sector": "MUEBLERO",
    "tipo_reclamacion": "Garantías",
    "procedimiento": "Conciliación",
    "bien_o_serv": "Bien",
    "medio_ingreso": "Escrito",
    "tipo_prod": "Mueble",
    "modalidad_compra": "Local comercial",
    "modalidad_pago": "Contado",
    "prob_especial": "NO",
    "costo_bien_servicio": 5000.0,
    "monto_reclamado": 5000.0,
    "monto_recuperado": 0.0,
    "dias_resolucion": 30.0,
}

resultado = predict_claim(ejemplo, artifacts)
print(f"Predicción: {resultado['prediccion']}")
print(f"Confianza: {resultado['confianza']:.2%}")
print(f"\nTop 5 probabilidades:")
for i, (cat, prob) in enumerate(resultado['probabilidades'].items()):
    if i >= 5:
        break
    print(f"  {cat}: {prob:.2%}")

In [ ]:
print("\n" + "=" * 60)
print("  PIPELINE COMPLETADO EXITOSAMENTE")
print("=" * 60)
print(f"\nModelo guardado en: {MODEL_PATH}")
print(f"Métricas guardadas en: {METRICS_PATH}")
print(f"\nPara exponer el modelo como API, ejecuta:")
print(f"  cd clasificador-reclamos-backend")
print(f"  uvicorn server:app --reload")

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# === CONFIGURACIÓN ===
DATA_PATH       = "quejas_limpias.csv"   # ajusta si tu ruta es diferente en Colab
TEXT_MODEL_PATH = "clasificador-reclamos-backend/models/text_model.pkl"
TARGET              = "MOTIVO_RECLAMACION"
MIN_CATEGORY_COUNT  = 30
TEST_SIZE           = 0.2
RANDOM_STATE        = 42

# Cargar datos
df_text = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# Agrupar categorías con pocos registros
counts_t   = df_text[TARGET].value_counts()
low_cats_t = counts_t[counts_t < MIN_CATEGORY_COUNT].index.tolist()
df_text[TARGET] = df_text[TARGET].apply(lambda x: "Otros" if x in low_cats_t else x)

# Crear texto: concatenar columnas descriptivas
df_text["texto"] = (
    df_text["TIPO_RECLAMACION"].fillna("") + " " +
    df_text["SECTOR"].fillna("") + " " +
    df_text["BIEN O SERV"].fillna("") + " " +
    df_text["PROCEDIMIENTO"].fillna("")
).str.lower().str.strip()

X_txt = df_text["texto"]
y_txt = df_text[TARGET]

X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    X_txt, y_txt, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_txt
)

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=8000, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=1000, class_weight="balanced",
                                  random_state=RANDOM_STATE, C=5.0,
                                  solver="lbfgs", multi_class="multinomial")),
])

text_pipeline.fit(X_txt_train, y_txt_train)

y_txt_pred = text_pipeline.predict(X_txt_test)
print(f"Accuracy:    {accuracy_score(y_txt_test, y_txt_pred):.4f}")
print(f"F1 weighted: {f1_score(y_txt_test, y_txt_pred, average='weighted'):.4f}")

os.makedirs(os.path.dirname(TEXT_MODEL_PATH), exist_ok=True)
joblib.dump(text_pipeline, TEXT_MODEL_PATH)
print(f"Modelo guardado en: {TEXT_MODEL_PATH}")